# Jigsaw Multilingual Toxic Comment Classification

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model, Sequential
from transformers import DistilBertTokenizerFast

In [2]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
    print('Running on TPU:', tpu.master())
except ValueError:
    strategy = tf.distribute.get_strategy()
    print('Running on CPU/GPU')

Running on CPU/GPU


In [3]:
strategy.num_replicas_in_sync

1

In [4]:
train1 = pd.read_csv("jigsaw-toxic-comment-train.csv")
valid = pd.read_csv('validation.csv')
test = pd.read_csv('test.csv')

In [5]:
train1 = train1.loc[:12000, :]
y_train = train1.toxic.values
y_valid = valid.toxic.values

In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-multilingual-cased')

In [7]:
def fast_encode(texts, tokenizer, chunk_size=256, maxlen=192):
    all_ids = []
    for i in tqdm(range(0, len(texts), chunk_size)):
        text_chunk = texts[i:i+chunk_size].tolist()
        encs = tokenizer(
            text_chunk,
            max_length=maxlen,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,
            return_token_type_ids=False
        )
        all_ids.extend(encs['input_ids'])
    return np.array(all_ids)

In [8]:
x_train = fast_encode(train1.comment_text.astype(str), tokenizer, maxlen=192)
x_valid = fast_encode(valid.comment_text.astype(str), tokenizer, maxlen=192)
x_test = fast_encode(test.content.astype(str), tokenizer, maxlen=192)

100%|████████████████████████████████████████████████████████████████████████████████| 250/250 [00:13<00:00, 18.39it/s]


In [9]:
AUTO = tf.data.experimental.AUTOTUNE
EPOCHS = 2
BATCH_SIZE = 16 * strategy.num_replicas_in_sync

In [10]:
train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((x_train, y_train))
    .repeat()
    .shuffle(2048)
    .batch(BATCH_SIZE)
    .prefetch(AUTO)
)

In [11]:
valid_dataset = (
    tf.data.Dataset
    .from_tensor_slices((x_valid, y_valid))
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(AUTO)
)

In [12]:
with strategy.scope():
    model = Sequential([
        Input(shape=(192,), dtype=tf.int32),
        Embedding(input_dim=tokenizer.vocab_size, output_dim=128, input_length=192),
        SpatialDropout1D(0.3),
        LSTM(128, dropout=0.3, recurrent_dropout=0.3),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

C:\Users\CASPER\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 192, 128)            │      15,302,016 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d (SpatialDropout1D) │ (None, 192, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 15,433,729 (58.88 MB)

 Trainable params: 15,433,729 (58.88 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
n_steps = x_train.shape[0] // BATCH_SIZE
train_history = model.fit(
    train_dataset,
    steps_per_epoch=n_steps if n_steps > 0 else 1,
    validation_data=valid_dataset,
    epochs=EPOCHS
)

Epoch 1/2


C:\Users\CASPER\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


750/750 ━━━━━━━━━━━━━━━━━━━━ 360s 470ms/step - accuracy: 0.9047 - loss: 0.3209 - val_accuracy: 0.8462 - val_loss: 0.4551
Epoch 2/2
750/750 ━━━━━━━━━━━━━━━━━━━━ 351s 468ms/step - accuracy: 0.9063 - loss: 0.3117 - val_accuracy: 0.8462 - val_loss: 0.4412


In [14]:
model.save('toxic_lstm_model.h5')
tokenizer.save_pretrained('distilbert_toxic_model')

('distilbert_toxic_model\\tokenizer_config.json',
 'distilbert_toxic_model\\tokenizer.json')